# Dyslexia Difficulty Dataset Builder
Builds a 42K-word dataset with 15 linguistic features and a composite `dyslexia_difficulty` label (0–10).
Output: `df_balanced` ready for model training, plus `ortho_n_lookup.pkl`, `bigram_freq_lookup.pkl`, `scaler.pkl`, and `difficulty_config_v2.json`.

In [ ]:
# ── Cell 1: Installs ──────────────────────────────────────────────────────────
!pip install -q wordfreq nltk pyphen scikit-learn pandas openpyxl tqdm transformers torch

In [ ]:
# ── Cell 2: Imports + Constants ───────────────────────────────────────────────
import os, re, json, pickle
import numpy as np
import pandas as pd
import pyphen
from collections import Counter
from functools import lru_cache
from tqdm import tqdm
tqdm.pandas()

import nltk
nltk.download('wordnet',  quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('cmudict',  quiet=True)
from nltk.corpus import wordnet, cmudict
from wordfreq import zipf_frequency, top_n_list
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# ── Paths ─────────────────────────────────────────────────────────────────────
SUBTLEX_PATH      = 'SUBTLEX-US frequency list with PoS and Zipf information.csv'
AOA_PATH          = 'AoA_51715_words.xlsx'
CONCRETENESS_PATH = 'concreteness.txt'

# ── Constants ─────────────────────────────────────────────────────────────────
STOP_WORDS = {
    'the','a','an','and','or','but','in','on','at','to','for',
    'of','with','by','from','is','was','are','were','be','been',
    'being','have','has','had','do','does','did','will','would',
    'could','should','may','might','shall','can','need','dare',
    'this','that','these','those','it','its','they','them','their',
    'he','she','we','you','i','my','your','his','her','our',
    'not','no','nor','so','yet','both','either','neither',
    'as','if','then','than','when','while','although','because',
    'into','onto','upon','about','above','below','between','through'
}

SILENT_PATTERNS    = ['kn','wr','gh','mb','bt','mn','cht','lm','sw','gn','ps','rh']
IRREGULAR_PATTERNS = ['ough','aigh','eigh','tion','sion','olo','queue','quay',
                       'eur','ieu','eau','ph','sch','chr']

POS_MAP = {
    'Noun':11,'Verb':12,'Adjective':13,'Adverb':14,
    'Preposition':5,'Conjunction':6,'Pronoun':7,
    'Article':8,'Determiner':9,'Number':10,
    'Interjection':11,'Name':12,'Unknown':0
}

LINGUISTIC_COLS = [
    'word_length', 'syllable_count', 'confusable_letters',
    'vowel_ratio', 'silent_letter_count', 'irregular_grapheme_count',
    'consonant_clusters', 'nphon', 'ortho_n', 'avg_bigram_freq',
    'morpheme_count', 'zipf_score', 'aoa', 'concreteness', 'pos_code'
]
N_LINGUISTIC = len(LINGUISTIC_COLS)   # 15

# ── Shared objects ────────────────────────────────────────────────────────────
dic = pyphen.Pyphen(lang='en')
cmu = cmudict.dict()

subtlex    = pd.read_csv(SUBTLEX_PATH)
aoa_df     = pd.read_excel(AOA_PATH)
conc_df    = pd.read_csv(CONCRETENESS_PATH, sep='\t')

subtlex_pos = subtlex.set_index('Word')['Dom_PoS_SUBTLEX'].to_dict()
aoa_lookup  = aoa_df.set_index('Word')['AoA_Kup'].to_dict()
conc_lookup = conc_df.set_index('Word')['Conc.M'].to_dict()

print(f"SUBTLEX: {len(subtlex):,} rows")
print(f"AoA:     {len(aoa_df):,} rows")
print(f"Concreteness: {len(conc_df):,} rows")
print(f"Linguistic features: {N_LINGUISTIC}")

In [ ]:
# ── Cell 3: Build Word List ───────────────────────────────────────────────────
# Three vocabulary sources; union gives the broadest coverage.
# All filtered to: alpha-only, length >= 3, zipf >= 1.5, not a stop word.

def _clean_source(words, zipf_threshold=1.5):
    return {
        str(w).lower() for w in words
        if isinstance(w, str)
        and w.isalpha()
        and len(w) >= 3
        and zipf_frequency(w.lower(), 'en') >= zipf_threshold
    }

subtlex_clean   = _clean_source(subtlex['Word'].dropna())
wordnet_clean   = _clean_source(wordnet.words())
brysbaert_clean = _clean_source(conc_df['Word'].dropna())

union = subtlex_clean | wordnet_clean | brysbaert_clean
union -= STOP_WORDS

print(f"SUBTLEX:    {len(subtlex_clean):,}")
print(f"WordNet:    {len(wordnet_clean):,}")
print(f"Brysbaert:  {len(brysbaert_clean):,}")
print(f"Union:      {len(union):,}")

# ── Stratified sampling to avoid very_rare dominating ─────────────────────────
# Target: ~42K words with balanced zipf strata
df = pd.DataFrame({'Word': sorted(union)})
df['zipf_score'] = df['Word'].apply(lambda w: zipf_frequency(w, 'en'))

def _zipf_stratum(z):
    if z < 2.0:   return 'very_rare'
    elif z < 3.0: return 'rare'
    elif z < 4.0: return 'uncommon'
    elif z < 5.0: return 'moderate'
    else:         return 'common'

df['stratum'] = df['zipf_score'].apply(_zipf_stratum)

STRATUM_TARGETS = {
    'very_rare': 12000,
    'rare':      15000,
    'uncommon':  9000,   # keep all if fewer
    'moderate':  6000,   # keep all if fewer
    'common':    1000,   # keep all if fewer
}

sampled = []
for stratum, target in STRATUM_TARGETS.items():
    pool = df[df['stratum'] == stratum]
    sampled.append(pool if len(pool) <= target else pool.sample(n=target, random_state=42))
df_balanced = pd.concat(sampled).reset_index(drop=True)

# ── Augment with high-frequency easy words from wordfreq top-5000 ─────────────
# Ensures the easy end of the difficulty scale is well represented.
top_words  = top_n_list('en', 5000)
existing   = set(df_balanced['Word'].str.lower())
new_easy   = [
    w for w in top_words
    if w.isalpha() and len(w) >= 3
    and w.lower() not in existing
    and w.lower() not in STOP_WORDS
    and _zipf_stratum(zipf_frequency(w, 'en')) in ('moderate', 'common')
]
easy_df = pd.DataFrame({'Word': new_easy})
easy_df['zipf_score'] = easy_df['Word'].apply(lambda w: zipf_frequency(w, 'en'))
easy_df['stratum']    = easy_df['zipf_score'].apply(_zipf_stratum)
df_balanced = pd.concat([df_balanced, easy_df]).reset_index(drop=True)

# ── Final dedup + lowercase ────────────────────────────────────────────────────
df_balanced['Word'] = df_balanced['Word'].str.lower()
df_balanced = df_balanced.drop_duplicates(subset='Word').reset_index(drop=True)

print(f"\nFinal dataset: {len(df_balanced):,} words")
print(df_balanced['stratum'].value_counts())

In [ ]:
# ── Cell 4: Feature Functions (all final versions) ────────────────────────────

def count_syllables_final(word):
    """CMU dict → pyphen → vowel-group fallback."""
    w = word.lower().strip()
    if not w:
        return 1
    # CMU: count vowel phonemes (those ending in a digit)
    entries = cmu.get(w)
    if entries:
        return max(sum(1 for ph in entries[0] if ph[-1].isdigit()), 1)
    # pyphen
    try:
        result = len(dic.inserted(w).split('-'))
        if result > 1:
            return result
    except Exception:
        pass
    # vowel-group fallback — treat consonant-flanked y as vowel
    temp = re.sub(r'([^aeiou])y([^aeiou])', r'\1i\2', w)
    temp = re.sub(r'([^aeiou])y$', r'\1i', temp)
    groups = re.findall(r'[aeiou]+', temp)
    count  = len(groups)
    # silent e: consonant-consonant-e at end
    if (w.endswith('e') and len(w) > 3
            and w[-2] not in 'aeiou'
            and w[-3] not in 'aeiou'
            and count > 1):
        count -= 1
    return max(count, 1)


def count_nphon_robust(word):
    """CMU dict phoneme count → grapheme-to-phoneme approximation."""
    entries = cmu.get(word.lower())
    if entries:
        return len(entries[0])
    w   = word.lower()
    # multi-char sequences that map to a single phoneme
    multi = ['ough','aigh','eigh','tion','sion','ph','th','ch',
             'sh','wh','ck','ng','qu','gh','kn','wr']
    count, i = 0, 0
    while i < len(w):
        matched = False
        for pat in sorted(multi, key=len, reverse=True):
            if w[i:i+len(pat)] == pat:
                count  += 1
                i      += len(pat)
                matched = True
                break
        if not matched:
            if w[i] not in ('e',) or i < len(w) - 1:
                count += 1
            i += 1
    return max(count, 1)


def count_consonant_clusters(word):
    return len(re.findall(r'[bcdfghjklmnpqrstvwxyz]{2,}', word.lower()))


def count_morphemes(word):
    PREFIXES = ['un','re','pre','mis','dis','over','under','out','up']
    SUFFIXES = ['tion','sion','ness','ment','ity','ous','ful','less',
                'ing','ed','er','est','ly','al','ic','ize','ise',
                'able','ible','ance','ence']
    w, count = word.lower(), 1
    for p in PREFIXES:
        if w.startswith(p) and len(w) - len(p) > 2:
            count += 1
            w = w[len(p):]
            break
    for s in SUFFIXES:
        if w.endswith(s) and len(w) - len(s) > 2:
            count += 1
            break
    return count


def get_pos(word):
    pos = subtlex_pos.get(word, subtlex_pos.get(word.lower(), 'Unknown'))
    if pd.isna(pos):
        pos = 'Unknown'
    return POS_MAP.get(pos, 0)


def get_context(word):
    synsets = wordnet.synsets(word)
    if not synsets:
        return word
    for s in synsets:
        if s.examples():
            return s.examples()[0]
    return synsets[0].definition()


print("Feature functions defined.")

In [ ]:
# ── Cell 5: Compute All Features ──────────────────────────────────────────────
# ortho_n and avg_bigram_freq are computed here against the training vocabulary
# and saved as lookup dicts so inference doesn't need to recompute them.

word_set = set(df_balanced['Word'])

# ── avg_bigram_freq: precompute from training vocabulary ──────────────────────
print("Computing bigram frequencies...")
all_text      = ' '.join(df_balanced['Word'])
all_bigrams   = [all_text[i:i+2] for i in range(len(all_text)-1)
                 if ' ' not in all_text[i:i+2]]
bigram_counts = Counter(all_bigrams)
total_bigrams = sum(bigram_counts.values())

def _avg_bigram_freq(word):
    w = word.lower()
    if len(w) < 2:
        return 0.0
    bgs   = [w[i:i+2] for i in range(len(w)-1)]
    freqs = [bigram_counts.get(bg, 0) / total_bigrams for bg in bgs]
    return float(np.mean(freqs)) if freqs else 0.0

# ── ortho_n: single-letter substitutions within training vocabulary ────────────
print("Computing orthographic neighbourhood (takes ~1 min)...")
def _ortho_n(word):
    w, count = word.lower(), 0
    for i in range(len(w)):
        for c in 'abcdefghijklmnopqrstuvwxyz':
            if c != w[i] and (w[:i] + c + w[i+1:]) in word_set:
                count += 1
    return count

# ── Apply all features ────────────────────────────────────────────────────────
print("Computing scalar features...")
df_balanced['word_length']              = df_balanced['Word'].str.len()
df_balanced['syllable_count']           = df_balanced['Word'].progress_apply(count_syllables_final)
df_balanced['confusable_letters']       = df_balanced['Word'].apply(
    lambda w: sum(w.lower().count(c) for c in 'bdpq'))
df_balanced['vowel_ratio']              = df_balanced['Word'].apply(
    lambda w: sum(1 for c in w.lower() if c in 'aeiou') / len(w) if len(w) > 0 else 0)
df_balanced['silent_letter_count']      = df_balanced['Word'].apply(
    lambda w: sum(p in w.lower() for p in SILENT_PATTERNS))
df_balanced['irregular_grapheme_count'] = df_balanced['Word'].apply(
    lambda w: sum(p in w.lower() for p in IRREGULAR_PATTERNS))
df_balanced['consonant_clusters']       = df_balanced['Word'].apply(count_consonant_clusters)
df_balanced['nphon']                    = df_balanced['Word'].progress_apply(count_nphon_robust)
df_balanced['morpheme_count']           = df_balanced['Word'].apply(count_morphemes)
df_balanced['avg_bigram_freq']          = df_balanced['Word'].apply(_avg_bigram_freq)
df_balanced['pos_code']                 = df_balanced['Word'].apply(get_pos)
df_balanced['ortho_n']                  = df_balanced['Word'].progress_apply(_ortho_n)

# ── AoA: real values where available, linear regression estimate elsewhere ─────
print("Filling AoA...")
df_balanced['aoa'] = df_balanced['Word'].apply(lambda w: aoa_lookup.get(w, np.nan))
have_aoa    = df_balanced['aoa'].notna()
X_aoa       = df_balanced.loc[have_aoa, ['zipf_score']].values
y_aoa       = df_balanced.loc[have_aoa, 'aoa'].values
aoa_reg     = LinearRegression().fit(X_aoa, y_aoa)
missing_aoa = ~have_aoa
df_balanced.loc[missing_aoa, 'aoa'] = aoa_reg.predict(
    df_balanced.loc[missing_aoa, ['zipf_score']].values
).clip(1.0, 17.5)
df_balanced['aoa'] = df_balanced['aoa'].clip(1.0, 17.5)
print(f"  AoA real: {have_aoa.sum():,}  estimated: {missing_aoa.sum():,}")

# ── Concreteness: real values where available, regression estimate elsewhere ───
print("Filling concreteness...")
df_balanced['concreteness'] = df_balanced['Word'].apply(
    lambda w: conc_lookup.get(w, np.nan))
have_conc    = df_balanced['concreteness'].notna()
X_conc       = df_balanced.loc[have_conc,
    ['zipf_score','word_length','syllable_count','pos_code']].values
y_conc       = df_balanced.loc[have_conc, 'concreteness'].values
conc_reg     = LinearRegression().fit(X_conc, y_conc)
missing_conc = ~have_conc
df_balanced.loc[missing_conc, 'concreteness'] = conc_reg.predict(
    df_balanced.loc[missing_conc,
        ['zipf_score','word_length','syllable_count','pos_code']].values
).clip(1.0, 5.0)
print(f"  Concreteness real: {have_conc.sum():,}  estimated: {missing_conc.sum():,}")

# ── Verify no NaNs ────────────────────────────────────────────────────────────
nan_counts = df_balanced[LINGUISTIC_COLS].isna().sum()
if nan_counts.sum() == 0:
    print("\nAll 15 features: clean (no NaNs)")
else:
    print("\nNaN check FAILED:")
    print(nan_counts[nan_counts > 0])

In [ ]:
# ── Cell 6: Label Formula v3 + Validation ─────────────────────────────────────
# Key design: frequency dampens orthographic penalty via familiarity_factor.
# Common irregular words (knight, though) are easier because readers
# recognise them as whole units — the interaction term models this.

def compute_dyslexia_difficulty(row):
    # ── Decoding
    irregular  = min(float(row['irregular_grapheme_count']) / 2.0, 1.0)
    silent     = min(float(row['silent_letter_count'])      / 2.0, 1.0)
    clusters   = min(float(row['consonant_clusters'])        / 2.0, 1.0)
    confusable = min(float(row['confusable_letters'])        / 2.0, 1.0)
    syllables  = min((float(row['syllable_count']) - 1.0)    / 4.0, 1.0)
    nphon_diff = min((float(row['nphon']) - 1.0)             / 7.0, 1.0)

    # ── Visual
    length     = min((float(row['word_length']) - 2.0)  / 10.0, 1.0)
    vowel_diff = 1.0 - min(float(row['vowel_ratio']), 1.0)

    # ── Orthographic familiarity
    bigram_diff = 1.0 - min(float(row['avg_bigram_freq']) / 0.03, 1.0)
    ortho_diff  = 1.0 - min(float(row['ortho_n'])         / 8.0,  1.0)

    # ── Morphological
    morphemes  = min((float(row['morpheme_count']) - 1.0) / 2.0, 1.0)

    # ── Familiarity
    zipf       = float(row['zipf_score'])
    freq_diff  = 1.0 - min(zipf / 6.0, 1.0)
    aoa_diff   = min((float(row['aoa']) - 1.0) / 16.5, 1.0)
    conc_diff  = 1.0 - min(float(row['concreteness']) / 5.0, 1.0)

    # ── Interaction: frequency dampens orthographic penalty ──────────────
    # At zipf=6 (very common): factor=0.60 → orthographic contribution halved
    # At zipf=1.5 (rare):      factor=0.90 → nearly full orthographic penalty
    familiarity_factor     = 1.0 - (min(zipf / 6.0, 1.0) * 0.4)
    orthographic_component = (
        irregular  * 0.12 +
        silent     * 0.08 +
        clusters   * 0.10 +
        confusable * 0.06
    ) * familiarity_factor

    score = (
        orthographic_component +
        syllables  * 0.09 +
        nphon_diff * 0.04 +
        length     * 0.07 +
        vowel_diff * 0.07 +
        bigram_diff* 0.04 +
        ortho_diff * 0.03 +
        morphemes  * 0.04 +
        freq_diff  * 0.15 +
        aoa_diff   * 0.07 +
        conc_diff  * 0.06
    )
    return round(score * 10.0, 4)

print("Computing labels...")
df_balanced['dyslexia_difficulty'] = df_balanced.apply(
    compute_dyslexia_difficulty, axis=1
)

# ── Distribution ──────────────────────────────────────────────────────────────
d = df_balanced['dyslexia_difficulty']
print(f"\nLabel distribution:")
print(f"  min={d.min():.3f}  max={d.max():.3f}  "
      f"mean={d.mean():.3f}  median={d.median():.3f}  std={d.std():.3f}")

easy   = (d < 2.5).sum()
medium = ((d >= 2.5) & (d < 5.0)).sum()
hard   = ((d >= 5.0) & (d < 7.0)).sum()
vhard  = (d >= 7.0).sum()
n      = len(df_balanced)
print(f"\n  Easy     (<2.5): {easy:,}  ({easy/n*100:.1f}%)")
print(f"  Medium (2.5-5):  {medium:,}  ({medium/n*100:.1f}%)")
print(f"  Hard   (5-7):    {hard:,}  ({hard/n*100:.1f}%)")
print(f"  V.Hard   (7+):   {vhard:,}  ({vhard/n*100:.1f}%)")

# ── Spot check ────────────────────────────────────────────────────────────────
spot = ['eat','cat','fire','table','knight','though','understand',
        'difficulty','environmental','straightforward',
        'jurisdiction','philosophy','nobleness']
print("\nSpot check:")
print(f"  {'word':25s} {'score':>6}  {'syll':>4}  {'zipf':>5}  {'irreg':>5}  {'silent':>6}")
for w in spot:
    row = df_balanced[df_balanced['Word'] == w]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"  {w:25s} {r['dyslexia_difficulty']:>6.3f}  "
              f"{int(r['syllable_count']):>4}  {r['zipf_score']:>5.2f}  "
              f"{int(r['irregular_grapheme_count']):>5}  "
              f"{int(r['silent_letter_count']):>6}")

df_sorted = df_balanced.sort_values('dyslexia_difficulty')
print("\nEasiest 10:")
print(df_sorted[['Word','dyslexia_difficulty','syllable_count','zipf_score']].head(10).to_string())
print("\nHardest 10:")
print(df_sorted[['Word','dyslexia_difficulty','syllable_count','zipf_score']].tail(10).to_string())

In [ ]:
# ── Cell 7: Scaler + Feature Matrix + Save Lookup Dicts ──────────────────────
# ortho_n and avg_bigram_freq lookups are saved here so difficulty_scorer.py
# can load them at startup instead of recomputing against a word list.

# ── Feature matrix ────────────────────────────────────────────────────────────
df_balanced[LINGUISTIC_COLS] = df_balanced[LINGUISTIC_COLS].fillna(0)
scaler      = StandardScaler()
ling_matrix = scaler.fit_transform(df_balanced[LINGUISTIC_COLS].values.astype(float))

print("Scaler fitted.")
print(f"  Feature matrix shape: {ling_matrix.shape}")
print(f"  Mean range:  [{scaler.mean_.min():.3f}, {scaler.mean_.max():.3f}]")
print(f"  Scale range: [{scaler.scale_.min():.3f}, {scaler.scale_.max():.3f}]")

# ── Lookup dicts ──────────────────────────────────────────────────────────────
ortho_n_lookup      = dict(zip(df_balanced['Word'], df_balanced['ortho_n'].astype(int)))
bigram_freq_lookup  = dict(zip(df_balanced['Word'], df_balanced['avg_bigram_freq']))

with open('ortho_n_lookup.pkl',     'wb') as f: pickle.dump(ortho_n_lookup,     f)
with open('bigram_freq_lookup.pkl', 'wb') as f: pickle.dump(bigram_freq_lookup, f)
with open('scaler.pkl',             'wb') as f: pickle.dump(scaler,             f)

# ── Additional lookups needed by difficulty_scorer.py at inference time ──────
pos_lookup          = dict(zip(df_balanced['Word'], df_balanced['pos_code'].astype(int)))
aoa_lookup_export   = dict(zip(df_balanced['Word'], df_balanced['aoa'].round(4)))
conc_lookup_export  = dict(zip(df_balanced['Word'], df_balanced['concreteness'].round(4)))

with open('pos_lookup.pkl',          'wb') as f: pickle.dump(pos_lookup,         f)
with open('aoa_lookup.pkl',          'wb') as f: pickle.dump(aoa_lookup_export,  f)
with open('concreteness_lookup.pkl', 'wb') as f: pickle.dump(conc_lookup_export, f)

print(f"\nSaved ortho_n_lookup.pkl      ({len(ortho_n_lookup):,} entries)")
print(f"Saved bigram_freq_lookup.pkl  ({len(bigram_freq_lookup):,} entries)")
print(f"Saved pos_lookup.pkl          ({len(pos_lookup):,} entries)")
print(f"Saved aoa_lookup.pkl          ({len(aoa_lookup_export):,} entries)")
print(f"Saved concreteness_lookup.pkl ({len(conc_lookup_export):,} entries)")
print(f"Saved scaler.pkl")

# ── Config ────────────────────────────────────────────────────────────────────
config = {
    'transformer_name': 'bert-base-uncased',
    'max_len':          64,
    'n_linguistic':     N_LINGUISTIC,
    'linguistic_cols':  LINGUISTIC_COLS,
    'label':            'dyslexia_difficulty',
    'label_min':        0.0,
    'label_max':        10.0,
    'scaler_mean':      scaler.mean_.tolist(),
    'scaler_scale':     scaler.scale_.tolist(),
}
with open('difficulty_config_v2.json', 'w') as f:
    json.dump(config, f, indent=2)
print("Saved difficulty_config_v2.json")

In [ ]:
# ── Cell 8: WordNet Context Sentences + Dataset / DataLoader ─────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split

TRANSFORMER_NAME = 'bert-base-uncased'
MAX_LEN          = 64
BATCH_SIZE       = 64
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

# ── Context sentences (cached) ────────────────────────────────────────────────
print("Building WordNet context sentences...")
df_balanced['context'] = df_balanced['Word'].progress_apply(get_context)
coverage = (df_balanced['context'] != df_balanced['Word']).mean() * 100
print(f"WordNet coverage: {coverage:.1f}%")

# ── Arrays for training ───────────────────────────────────────────────────────
words    = df_balanced['Word'].tolist()
contexts = df_balanced['context'].tolist()
labels   = df_balanced['dyslexia_difficulty'].values.astype(float)

# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_NAME)

# ── Dataset ──────────────────────────────────────────────────────────────────
class WordDifficultyDataset(Dataset):
    def __init__(self, words, contexts, ling_features, labels, tokenizer, max_len):
        self.words     = words
        self.contexts  = contexts
        self.ling      = ling_features
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.words[idx]),
            str(self.contexts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'linguistic':     torch.tensor(self.ling[idx], dtype=torch.float32),
            'label':          torch.tensor(self.labels[idx], dtype=torch.float32),
        }

# ── Train / val split ────────────────────────────────────────────────────────
idx_train, idx_val = train_test_split(
    np.arange(len(words)), test_size=0.2, random_state=42
)

train_ds = WordDifficultyDataset(
    [words[i] for i in idx_train], [contexts[i] for i in idx_train],
    ling_matrix[idx_train], labels[idx_train], tokenizer, MAX_LEN
)
val_ds = WordDifficultyDataset(
    [words[i] for i in idx_val], [contexts[i] for i in idx_val],
    ling_matrix[idx_val], labels[idx_val], tokenizer, MAX_LEN
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"\nTrain: {len(train_ds):,}  Val: {len(val_ds):,}")
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

In [ ]:
# ── Cell 9: Model + Training Loop ─────────────────────────────────────────────
from sklearn.metrics import r2_score, mean_squared_error

EPOCHS       = 20
LR_ENCODER   = 2e-5
LR_HEAD      = 1e-3
PATIENCE     = 4      # early stopping
SAVE_PATH    = 'difficulty_model_v2.pt'

# ── Model ─────────────────────────────────────────────────────────────────────
class WordDifficultyModel(nn.Module):
    def __init__(self, transformer_name, n_linguistic):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(transformer_name)
        hidden       = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden + n_linguistic, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids, attention_mask, linguistic):
        cls  = self.encoder(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.head(torch.cat([cls, linguistic], dim=1)).squeeze(-1)

model = WordDifficultyModel(TRANSFORMER_NAME, N_LINGUISTIC).to(DEVICE)

# Freeze bottom 4 layers (was 6 — allows more task-specific fine-tuning)
for i, layer in enumerate(model.encoder.encoder.layer):
    if i < 4:
        for p in layer.parameters():
            p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}  Total: {total:,}")

# ── Optimiser + scheduler ─────────────────────────────────────────────────────
optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': LR_ENCODER, 'weight_decay': 0.05},
    {'params': model.head.parameters(),    'lr': LR_HEAD,    'weight_decay': 1e-4},
])
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 5,
    num_training_steps=total_steps
)
loss_fn = nn.MSELoss()

# ── Training loop with early stopping ────────────────────────────────────────
best_val_r2   = -np.inf
best_val_mse  = np.inf
patience_left = PATIENCE

for epoch in range(1, EPOCHS + 1):
    # ── Train
    model.train()
    train_losses = []
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]"):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        ling  = batch['linguistic'].to(DEVICE)
        lbl   = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        preds = model(ids, mask, ling)
        if torch.isnan(preds).any(): continue
        loss = loss_fn(preds, lbl)
        if torch.isnan(loss): continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()
        train_losses.append(loss.item())

    # ── Validate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]  "):
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            ling  = batch['linguistic'].to(DEVICE)
            preds = model(ids, mask, ling)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['label'].numpy())

    preds_np  = np.nan_to_num(np.array(all_preds,  dtype=np.float32))
    labels_np = np.array(all_labels, dtype=np.float32)
    val_mse   = mean_squared_error(labels_np, preds_np)
    val_r2    = r2_score(labels_np, preds_np)
    print(f"  train_loss={np.mean(train_losses):.4f} | "
          f"val_MSE={val_mse:.4f} | val_R2={val_r2:.4f}")

    # ── Checkpoint + early stopping
    if val_r2 > best_val_r2:
        best_val_r2   = val_r2
        best_val_mse  = val_mse
        patience_left = PATIENCE
        torch.save({
            'model_state_dict': model.state_dict(),
            'n_linguistic':     N_LINGUISTIC,
            'transformer_name': TRANSFORMER_NAME,
            'scaler_mean':      scaler.mean_.tolist(),
            'scaler_scale':     scaler.scale_.tolist(),
        }, SAVE_PATH)
        print(f"  ✓ Saved (R2={best_val_r2:.4f})")
    else:
        patience_left -= 1
        print(f"  No improvement. Patience: {patience_left}/{PATIENCE}")
        if patience_left == 0:
            print(f"  Early stopping at epoch {epoch}.")
            break

print(f"\nFinal best — MSE: {best_val_mse:.4f} | R2: {best_val_r2:.4f}")

# ── Download outputs ──────────────────────────────────────────────────────────
from google.colab import files
for f_name in [SAVE_PATH, 'difficulty_config_v2.json',
               'ortho_n_lookup.pkl', 'bigram_freq_lookup.pkl', 'scaler.pkl',
               'pos_lookup.pkl', 'aoa_lookup.pkl', 'concreteness_lookup.pkl']:
    files.download(f_name)
    print(f"Downloaded {f_name}")